# Hands-On LangChain #6: Answering Questions From Documents

Companion to [the full article](https://todatabeyond.com/blog/hands-on-langchain-for-llms-app-answering-questions-from-documents).

The default path runs real MiniLM embeddings, Chroma retrieval, and Google's pretrained FLAN-T5-small on CPU without API keys. First-run model downloads require internet. The three lecture passages below are short generated fixtures, NOT the original CS229 PDFs. This small model is a wiring demonstration, not a production answer-quality benchmark. Outputs can vary across library/model versions. All original hosted steps appear in the opt-in appendix; archived OpenAI outputs are historical and not asserted here.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-chroma>=1.0,<2" "langchain-classic>=1.0,<2" "langchain-community>=0.4,<0.5" "langchain-openai>=1.0,<2" "langchain-text-splitters>=1.0,<2" "transformers>=4.57,<5" torch sentencepiece pypdf numpy python-dotenv

## 2. Real local embedding and answer models

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
from langchain_core.embeddings import Embeddings
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

class LocalMiniLMEmbeddings(Embeddings):
    def __init__(self):
        self.model = ONNXMiniLM_L6_V2(preferred_providers=["CPUExecutionProvider"])
        self.model.DOWNLOAD_PATH = Path(tempfile.gettempdir()) / "tdb-minilm-model-cache"

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [vector.tolist() for vector in self.model(texts)]

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

embedding = LocalMiniLMEmbeddings()
print(f"Embedding dimensions: {len(embedding.embed_query('example'))}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM

torch.set_num_threads(2)
cache = str(Path(tempfile.gettempdir()) / "tdb-flan-t5-cache")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model.eval()
calls = []

class LocalAnswerLLM(LLM):
    @property
    def _llm_type(self): return "local-flan-t5-small"
    def get_num_tokens(self, text): return len(tokenizer.encode(text))
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=False)
        if inputs["input_ids"].shape[1] > 512:
            raise ValueError("Prompt exceeds this demo's 512-token input budget.")
        with torch.inference_mode():
            tokens = answer_model.generate(**inputs, max_new_tokens=64, do_sample=False)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True)
        calls.append({"prompt": prompt, "answer": answer})
        return answer

llm = LocalAnswerLLM()
print("Models: all-MiniLM-L6-v2 + google/flan-t5-small (CPU, no API key)")

## 3. Retrieve generated lecture fixtures

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

fixtures = [
    Document(page_content="The class teaches supervised learning, unsupervised learning, and reinforcement learning.", metadata={"source": "generated-lecture-1", "page": 0}),
    Document(page_content="The class requires prior knowledge of probability and statistics. Probability is a prerequisite, not a new core topic.", metadata={"source": "generated-lecture-1", "page": 1}),
    Document(page_content="Linear algebra is a prerequisite because machine learning uses vectors and matrices.", metadata={"source": "generated-lecture-2", "page": 0}),
]
directory = tempfile.mkdtemp(prefix="tdb-document-qa-")
db = Chroma.from_documents(fixtures, embedding=embedding, collection_name="qa_lectures", persist_directory=directory)
retriever = db.as_retriever(search_kwargs={"k": 3})
question = "What are the prerequisites for the class?"
retrieved = retriever.invoke(question)
assert len(retrieved) == 3
assert {d.metadata["source"] for d in retrieved} == {"generated-lecture-1", "generated-lecture-2"}
print(f"Retrieved passages: {len(retrieved)}")

## 4. Stuff: explicit retrieval and source-aware answering
`create_retrieval_chain` is a maintained explicit composition from langchain-classic. The returned context contains the exact retrieved documents. A model answer alone is not proof that it is supported by those sources.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain, RetrievalQA
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

prompt = PromptTemplate.from_template("Answer using this context only.\nContext: {context}\nQuestion: {input}\nAnswer:")
stuff = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))
calls.clear()
result = stuff.invoke({"input": question})
assert len(calls) == 1 and len(result["context"]) == 3
assert all(d.page_content in calls[0]["prompt"] for d in result["context"])
assert result["answer"] and result["answer"] == calls[0]["answer"]
print("Stuff answer:", result["answer"])
print("Source passages:", len(result["context"]))
print("Stuff model calls:", len(calls))

## 5. Map-reduce: answer each passage then combine
Short custom prompts keep this small model inside its input budget. Map-reduce is not guaranteed to outperform stuff; independent map calls can lose context.

In [ ]:
map_prompt = PromptTemplate.from_template("Context: {context}\nQuestion: {question}\nAnswer from the context:")
combine_prompt = PromptTemplate.from_template("Combine these partial answers to answer the question.\nPartial answers: {summaries}\nQuestion: {question}\nAnswer:")
map_reduce = RetrievalQA.from_chain_type(llm, retriever=retriever, chain_type="map_reduce", return_source_documents=True, chain_type_kwargs={"question_prompt": map_prompt, "combine_prompt": combine_prompt, "token_max": 512})
calls.clear()
mr = map_reduce.invoke({"query": question})
assert len(calls) == 4 and len(mr["source_documents"]) == 3
assert all(c["answer"] in calls[-1]["prompt"] for c in calls[:-1])
print("Map-reduce answer:", mr["result"])
print("Map-reduce model calls:", len(calls))

## 6. Refine: revise an answer using each next passage
Refine is sequential and sensitive to document order; an early mistake can persist. These assertions verify the previous answer enters each next prompt, not that refinements always improve it.

In [ ]:
initial = PromptTemplate.from_template("Context: {context_str}\nQuestion: {question}\nAnswer:")
refine_prompt = PromptTemplate.from_template("Improve the existing answer using the new context.\nExisting answer: {existing_answer}\nNew context: {context_str}\nQuestion: {question}\nAnswer:")
refine = RetrievalQA.from_chain_type(llm, retriever=retriever, chain_type="refine", return_source_documents=True, chain_type_kwargs={"question_prompt": initial, "refine_prompt": refine_prompt})
calls.clear()
refined = refine.invoke({"query": question})
assert len(calls) == 3
assert all(calls[i-1]["answer"] in calls[i]["prompt"] for i in range(1, 3))
print("Refine answer:", refined["result"])
print("Refine model calls:", len(calls))

## 7. Stateless follow-up and persistence
An ordinary retrieval chain does not remember prior questions. Test this by inspecting the actual model prompt, not by guessing whether a follow-up answer sounds correct.

In [ ]:
calls.clear()
first = stuff.invoke({"input": "Is probability a prerequisite?"})
follow_up = stuff.invoke({"input": "Why are those prerequisites needed?"})
assert len(calls) == 2
assert "Is probability a prerequisite?" not in calls[-1]["prompt"]
assert "Existing answer:" not in calls[-1]["prompt"]
reopened = Chroma(collection_name="qa_lectures", embedding_function=embedding, persist_directory=directory)
assert len(reopened.get()["ids"]) == 3
print("Stateless follow-up: prior question is absent from the next model prompt")
print("Chroma persistence: 3 passages after reopening")
print("All local document-QA integration assertions passed.")

## 8. Complete opt-in hosted workflow

Every original tutorial code step follows, updated for current packages and `.invoke`. Set `OPENAI_API_KEY` and `OPENAI_CHAT_MODEL` locally, provide the three CS229 PDFs under `docs/cs229_lectures/`, then enable the flag. A fresh index prevents mixing different embedding models. Calls incur charges. This appendix is syntax-checked, not live-tested. `RetrievalQA` is a legacy API in langchain-classic, retained to match the archive; use the explicit composition above for new code. Neither temperature zero nor the prompt guarantees factual accuracy. Archived outputs can differ, including incorrect claims that a prerequisite is a core course topic.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from pathlib import Path
    from tempfile import mkdtemp
    from dotenv import load_dotenv
    load_dotenv()
    for name in ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"):
        if not os.environ.get(name):
            raise RuntimeError(f"Set {name} locally before running hosted examples.")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # Keep the duplicated first lecture from the original series corpus.
    pdf_paths = [f"docs/cs229_lectures/MachineLearning-Lecture{i:02d}.pdf" for i in (1, 1, 2, 3)]
    if not all(Path(path).is_file() for path in pdf_paths):
        raise FileNotFoundError("Place the three CS229 PDFs under docs/cs229_lectures/.")
    lecture_docs = []
    for path in pdf_paths:
        lecture_docs.extend(PyPDFLoader(path).load())
    splits = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150).split_documents(lecture_docs)
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")
    persist_directory = mkdtemp(prefix="langchain-document-qa-")
    vectordb = Chroma.from_documents(splits, embedding=embedding, collection_name="lectures", persist_directory=persist_directory)
    print(len(vectordb.get()["ids"]))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "What are major topics for this class?"
    docs = vectordb.similarity_search(question,k=3)
    print(len(docs))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_openai import ChatOpenAI
    llm_name = os.environ["OPENAI_CHAT_MODEL"]
    llm = ChatOpenAI(model=llm_name, temperature=0)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.chains import RetrievalQA
    qa_chain = RetrievalQA.from_chain_type(
        llm,
        retriever=vectordb.as_retriever()
    )
    
    result = qa_chain.invoke({"query": question})
    print(result["result"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_core.prompts import PromptTemplate
    
    # Build prompt
    template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer. 
    {context}
    Question: {question}
    Helpful Answer:"""
    QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # Run chain
    qa_chain = RetrievalQA.from_chain_type(
        llm,
        retriever=vectordb.as_retriever(),
        return_source_documents=True,
        chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "Is probability a class topic?"
    result = qa_chain.invoke({"query": question})
    print(result["result"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(result["source_documents"][0])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    qa_chain_mr = RetrievalQA.from_chain_type(
        llm,
        retriever=vectordb.as_retriever(),
        chain_type="map_reduce"
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    result = qa_chain_mr.invoke({"query": question})
    print(result["result"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    qa_chain_mr = RetrievalQA.from_chain_type(
        llm,
        retriever=vectordb.as_retriever(),
        chain_type="refine"
    )
    result = qa_chain_mr.invoke({"query": question})
    print(result["result"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    qa_chain = RetrievalQA.from_chain_type(
        llm,
        retriever=vectordb.as_retriever()
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "Is probability a class topic?"
    result = qa_chain.invoke({"query": question})
    print(result["result"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "why are those prerequesites needed?"
    result = qa_chain.invoke({"query": question})
    print(result["result"])